# 第 7 章 · shared memory 与 bank conflict

本章回答：shared memory 怎么声明、怎么在 block 内交换数据？`__syncthreads()` 为什么必不可少？shared memory 为什么也有"慢路"——什么是 bank conflict、怎么算几路冲突、怎么用 padding 消掉？

配套讲义：仓库里 `cuda-programming/notes/lessons/07-shared-memory-banks.md`（讲解、动画、练习都在讲义里）。对应原文：PG §2.3.3.2 Shared Memory（`materials/text/cuda-programming-guide.txt` 第 3260–3435 行）、§2.3.2.1 Thread Block Synchronization（第 3174–3208 行）、§2.3.4.2 Shared Memory Access Patterns（第 3930–4273 行，含转置例子与 bank conflict）；Best Practices §10.2.3.1（`text/cuda-c-best-practices-guide.txt` 第 1745–1771 行）；Oxford lec3

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import math

import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 写完再读 | `__syncthreads()` | `race_demo(True)` / `race_demo(False)` | 有栅栏全对；没栅栏前一半读到 −1（还没写） |
| 第三、四步 跨步 | bank = $w \bmod 32$，几路 = $\gcd(s, 32)$ | `conflict_ways(strided(s))` | s = 1、2、3、4、32 → 1、2、1、4、32 |
| 第三步 广播 | 同一地址 | `conflict_ways(np.zeros(32))` | 1 路 |
| 第四步 按列读 | `r * 32 + c` | `conflict_ways(tile_addr(r, 5, 32))` | 32 路 |
| 第五步 padding | `r * 33 + c` | `conflict_ways(tile_addr(r, 5, 33))` | 1 路，多用 3% |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `WARP`**（来自 `ch07_bank_conflict.py`）

In [ ]:
WARP = 32

**常量 `BANKS`**（来自 `ch07_bank_conflict.py`）

In [ ]:
BANKS = 32

**函数 `bank`**（来自 `ch07_bank_conflict.py`）：字地址（以 4 字节为单位）→ bank 号。

In [ ]:
def bank(word_addr):
    """字地址（以 4 字节为单位）→ bank 号。"""
    return np.asarray(word_addr) % BANKS

**函数 `conflict_ways`**（来自 `ch07_bank_conflict.py`）：几路冲突 = 同一个 bank 上不同地址最多有几个（同一地址算广播，只算一个）。

In [ ]:
def conflict_ways(word_addrs) -> int:
    """几路冲突 = 同一个 bank 上不同地址最多有几个（同一地址算广播，只算一个）。"""
    distinct = np.unique(np.asarray(word_addrs))
    return int(np.bincount(bank(distinct), minlength=BANKS).max())

**函数 `race_demo`**（来自 `ch07_bank_conflict.py`）：第二步的例子：s[t] = g[t]；（栅栏）；y = s[n-1-t]。

In [ ]:
def race_demo(with_barrier: bool, n: int = 8) -> np.ndarray:
    """第二步的例子：s[t] = g[t]；（栅栏）；y = s[n-1-t]。

    没有栅栏时模拟最坏的交错：每个线程写完自己那格马上去读，后面的线程还没写。
    """
    g = np.arange(10, 10 + n)
    s = np.full(n, -1)                    # -1 表示还没写进去的"垃圾值"
    y = np.zeros(n, dtype=int)
    if with_barrier:
        s[:] = g                          # ① 所有人先写
        y[:] = s[n - 1 - np.arange(n)]    # ③ 栅栏之后再读
    else:
        for t in range(n):                # 线程 0 先写先读，线程 1 再写再读……
            s[t] = g[t]
            y[t] = s[n - 1 - t]
    return y

**函数 `strided`**（来自 `ch07_bank_conflict.py`）：warp 里线程 t 读 s[t * stride]。

In [ ]:
def strided(stride: int) -> np.ndarray:
    """warp 里线程 t 读 s[t * stride]。"""
    return np.arange(WARP) * stride

**函数 `tile_addr`**（来自 `ch07_bank_conflict.py`）：二维 shared 数组 tile[r][c] 的字地址：r * 每行长度 + c。

In [ ]:
def tile_addr(r, c, row_len: int):
    """二维 shared 数组 tile[r][c] 的字地址：r * 每行长度 + c。"""
    return np.asarray(r) * row_len + np.asarray(c)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 __syncthreads()：写完再读")
print("  有栅栏：", race_demo(True))
print("  没栅栏：", race_demo(False), "（-1 是读到了还没写的位置）")

In [ ]:
print("\n第四步 跨步访问 s[t * stride]")
for s in (1, 2, 3, 4, 32):
    print(f"  stride {s:2d}: 用到 {len(np.unique(bank(strided(s)))):2d} 个 bank，"
          f"{conflict_ways(strided(s)):2d} 路冲突，gcd(stride, 32) = {math.gcd(s, 32)}")
print("  32 个线程都读 s[0]（同一地址）：", conflict_ways(np.zeros(WARP, dtype=int)), "路——广播")

In [ ]:
print("\n第四、五步 二维数组按列读：warp 的线程 r = 0..31 读 tile[r][5]")
r = np.arange(WARP)
print("  tile[32][32]：bank 都是", np.unique(bank(tile_addr(r, 5, 32))), "→",
      conflict_ways(tile_addr(r, 5, 32)), "路冲突")
print("  tile[32][33]：用到", len(np.unique(bank(tile_addr(r, 5, 33)))), "个 bank →",
      conflict_ways(tile_addr(r, 5, 33)), "路")
print("  按行读 tile[5][t]：", conflict_ways(tile_addr(5, r, 32)), "路")
print(f"  padding 多用 {32} 个 float，占 {32 / (32 * 32):.0%}")

## 3. 代码里才看得到的

1. **先 `np.unique` 再数**：同一个地址被几个线程同时读是广播，不算冲突；不去重的话"32 个线程都读 s[0]"会被错算成 32 路。
2. **`race_demo(False)` 是"最坏的交错"之一**：真实 GPU 上没有栅栏时读到什么取决于调度，可能碰巧全对，也可能全错——"碰巧全对"正是数据竞争难查的原因。

## 4. 自己改一改

- 把 `strided(s)` 的 s 换成 6、8、16，验证几路冲突 = gcd(s, 32)。
- 把 `tile_addr(r, 5, 33)` 的 33 换成 34、40，看 padding 多少才能完全不冲突。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> shared 是 block 的公共台面，写完要等、读完再覆盖也要等；
> `__syncthreads()` 要全员到场，别写在 if 里；
> 32 个窗口轮着排，字地址模 32 就是窗口号；
> 同窗不同菜要排队，同窗同菜一起打；冲突路数 = gcd(跨步, 32)，奇数跨步不冲突；
> 按列读二维数组冲突 32 路，每行多放一个就散开。